# 04 - BERT-TextCNN Multi-label Training for CTI ATT&CK Mapping

Notebook nay dung **deep learning thuan**, khong dung TF-IDF/Logistic/baseline ensemble.

Kien truc:

```text
DL_Text
-> SecureBERT2.0 tokenizer
-> SecureBERT2.0 encoder
-> last_hidden_state [batch, seq_len, hidden]
-> TextCNN Conv1D kernels [2, 3, 4, 5]
-> max pooling
-> MLP classifier
-> 108 multi-label logits
```

Ly do chon BERT-TextCNN:

- BERT/SecureBERT bo sung contextual semantics cho CTI narrative.
- TextCNN bat local technical patterns tren hidden states: command, path, API, payload, tool name.
- Van la deep learning thuan, khong tron voi baseline TF-IDF.

Kaggle paths:

```text
/kaggle/input/attack-dataset-for-dl/attack_dataset_dl_stage1_frequent.csv
/kaggle/input/attack-dataset-for-dl/special_tokens.json
/kaggle/working/artifacts_bert_textcnn_multilabel
```


In [ ]:
# Kaggle dependency cell
import subprocess
import sys

REQUIRED_PACKAGES = [
    "transformers>=4.48.0",
    "accelerate>=0.34.0",
    "iterative-stratification>=0.1.7",
    "scikit-learn>=1.3.0",
    "joblib>=1.3.0",
]

print("Installing/updating required packages if needed...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *REQUIRED_PACKAGES])
    print("Dependency installation finished.")
except Exception as exc:
    print("WARNING: pip install failed. On Kaggle, enable Internet or attach wheel datasets.")
    print(type(exc).__name__, exc)


In [ ]:
from pathlib import Path
import json
import math
import random
import time

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup, set_seed

try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception:
    pass
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Cau hinh

Mac dinh dung `cisco-ai/SecureBERT2.0-base` lam BERT encoder. Neu Kaggle bi OOM, giam `MAX_LENGTH` xuong 384 hoac giam `TRAIN_BATCH_SIZE` xuong 2.

`ADD_DOMAIN_SPECIAL_TOKENS=True` vi encoder duoc fine-tune, nen embedding cua token moi co co hoi duoc hoc. Neu thay overfit/dao dong, doi ve `False`.


In [ ]:
RANDOM_STATE = 42
MODEL_NAME = "cisco-ai/SecureBERT2.0-base"
TEXT_COL = "DL_Text"
LABEL_COL = "Labels"

MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 6
LEARNING_RATE_ENCODER = 1.0e-5
LEARNING_RATE_HEAD = 8.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
MAX_GRAD_NORM = 1.0
LOG_EVERY_STEPS = 10
EARLY_STOPPING_PATIENCE = 3

CNN_FILTER_SIZES = [2, 3, 4, 5]
CNN_NUM_FILTERS = 128
CNN_DROPOUT = 0.35
CLASSIFIER_HIDDEN_DIM = 512

ADD_DOMAIN_SPECIAL_TOKENS = True
FREEZE_ENCODER_EPOCHS = 0

USE_WEIGHTED_SAMPLER = False
RARE_WEIGHT_POWER = 0.5
RARE_WEIGHT_CAP = 6.0
SAMPLE_WEIGHT_CAP = 8.0

# Cau hinh goc cua BERT-TextCNN tot nhat. Chi doi encoder sang SecureBERT.
POS_WEIGHT_POWER = 0.5
POS_WEIGHT_CAP = 5.0

SOC_THRESHOLD = 0.35
MIN_K = 1
MAX_K = 3
THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KAGGLE_DATASET_DIR_CANDIDATES = [
    Path("/kaggle/input/attack-dataset-for-dl"),
    Path("/kaggle/input/attack_dataset_for_dl"),
]
KAGGLE_DATA_PATH_CANDIDATES = [d / "attack_dataset_dl_stage1_frequent.csv" for d in KAGGLE_DATASET_DIR_CANDIDATES]
KAGGLE_SPECIAL_TOKENS_PATH_CANDIDATES = [d / "special_tokens.json" for d in KAGGLE_DATASET_DIR_CANDIDATES]
LOCAL_DL_DIR = PROJECT_ROOT / "dataset" / "processed_deeplearning"
LOCAL_DATA_PATH = LOCAL_DL_DIR / "attack_dataset_dl_stage1_frequent.csv"
LOCAL_SPECIAL_TOKENS_PATH = LOCAL_DL_DIR / "special_tokens.json"
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
ARTIFACT_DIR = KAGGLE_WORKING / "artifacts_bert_textcnn_multilabel" if KAGGLE_WORKING.exists() else PROJECT_ROOT / "artifacts" / "bert_textcnn_multilabel"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

def first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

def find_in_kaggle(filename):
    if KAGGLE_INPUT.exists():
        matches = list(KAGGLE_INPUT.rglob(filename))
        if matches:
            return matches[0]
    return None

def resolve_data_path():
    explicit = first_existing_path(KAGGLE_DATA_PATH_CANDIDATES)
    if explicit is not None:
        return explicit
    if LOCAL_DATA_PATH.exists():
        return LOCAL_DATA_PATH
    auto = find_in_kaggle("attack_dataset_dl_stage1_frequent.csv")
    if auto is not None:
        return auto
    raise FileNotFoundError("Cannot find attack_dataset_dl_stage1_frequent.csv")

def resolve_special_tokens_path(data_path):
    explicit = first_existing_path(KAGGLE_SPECIAL_TOKENS_PATH_CANDIDATES)
    if explicit is not None:
        return explicit
    if LOCAL_SPECIAL_TOKENS_PATH.exists():
        return LOCAL_SPECIAL_TOKENS_PATH
    sibling = Path(data_path).parent / "special_tokens.json"
    if sibling.exists():
        return sibling
    return find_in_kaggle("special_tokens.json")

DATA_PATH = resolve_data_path()
SPECIAL_TOKENS_PATH = resolve_special_tokens_path(DATA_PATH)

print("Data path:", DATA_PATH)
print("Special tokens path:", SPECIAL_TOKENS_PATH)
print("Artifact dir:", ARTIFACT_DIR)
print("Device:", DEVICE)


## 2. Load dataset va split multi-label stratified

In [ ]:
df = pd.read_csv(DATA_PATH)
if TEXT_COL not in df.columns:
    if "Cleaned_Text" in df.columns:
        print(f"WARNING: {TEXT_COL} not found. Falling back to Cleaned_Text.")
        TEXT_COL = "Cleaned_Text"
    else:
        raise ValueError(f"Missing text column: {TEXT_COL}")

df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

def split_labels(label_str):
    return [label.strip() for label in str(label_str).split(",") if label.strip()]

df["label_list"] = df[LABEL_COL].apply(split_labels)
all_labels = sorted({label for labels in df["label_list"] for label in labels})
mlb = MultiLabelBinarizer(classes=all_labels)
Y_all = mlb.fit_transform(df["label_list"]).astype(np.float32)
label_classes = list(mlb.classes_)
num_labels = len(label_classes)

split_train_temp = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(split_train_temp.split(np.zeros(len(df)), Y_all))
split_val_test = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_relative_idx, test_relative_idx = next(split_val_test.split(np.zeros(len(temp_idx)), Y_all[temp_idx]))
val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
val_df = df.iloc[val_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)
Y_train = Y_all[train_idx]
Y_val = Y_all[val_idx]
Y_test = Y_all[test_idx]

summary_df = pd.DataFrame([
    {"split": "train", "samples": len(train_df), "min_label_support": int(Y_train.sum(axis=0).min()), "avg_labels": float(Y_train.sum(axis=1).mean())},
    {"split": "validation", "samples": len(val_df), "min_label_support": int(Y_val.sum(axis=0).min()), "avg_labels": float(Y_val.sum(axis=1).mean())},
    {"split": "test", "samples": len(test_df), "min_label_support": int(Y_test.sum(axis=0).min()), "avg_labels": float(Y_test.sum(axis=1).mean())},
])
print("Rows:", len(df))
print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Num labels:", num_labels)
print("Avg labels/sample:", float(Y_all.sum(axis=1).mean()))
display(summary_df)

joblib.dump(mlb, ARTIFACT_DIR / "multilabel_binarizer.joblib")
pd.Series(label_classes, name="label").to_csv(ARTIFACT_DIR / "label_classes.csv", index=False)


## 3. Tokenizer va PyTorch datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if ADD_DOMAIN_SPECIAL_TOKENS and SPECIAL_TOKENS_PATH is not None and Path(SPECIAL_TOKENS_PATH).exists():
    with open(SPECIAL_TOKENS_PATH, "r", encoding="utf-8") as f:
        special_tokens = json.load(f)
    added_tokens = tokenizer.add_special_tokens(special_tokens)
else:
    special_tokens = None
    added_tokens = 0

print("Tokenizer size:", len(tokenizer))
print("Added special tokens:", added_tokens)
print("MAX_LENGTH:", MAX_LENGTH)

class CTIMultilabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = labels.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors=None,
        )
        return {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

train_dataset = CTIMultilabelDataset(train_df[TEXT_COL].values, Y_train, tokenizer, MAX_LENGTH)
val_dataset = CTIMultilabelDataset(val_df[TEXT_COL].values, Y_val, tokenizer, MAX_LENGTH)
test_dataset = CTIMultilabelDataset(test_df[TEXT_COL].values, Y_test, tokenizer, MAX_LENGTH)

if USE_WEIGHTED_SAMPLER:
    train_label_freq = Y_train.sum(axis=0).astype(np.float32)
    rare_label_weight = (train_label_freq.max() / np.maximum(train_label_freq, 1.0)) ** RARE_WEIGHT_POWER
    rare_label_weight = np.clip(rare_label_weight, 1.0, RARE_WEIGHT_CAP).astype(np.float32)
    sample_weight = (Y_train * rare_label_weight.reshape(1, -1)).max(axis=1)
    sample_weight = np.clip(sample_weight, 1.0, SAMPLE_WEIGHT_CAP).astype(np.float64)
    train_sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weight),
        num_samples=len(sample_weight),
        replacement=True,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        sampler=train_sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    sampled_indices = list(iter(train_sampler))
    sampled_label_freq = Y_train[sampled_indices].sum(axis=0)
    sampler_report = pd.DataFrame({
        "label": label_classes,
        "original_train_freq": train_label_freq.astype(int),
        "sampled_epoch_freq": sampled_label_freq.astype(int),
        "exposure_ratio": sampled_label_freq / np.maximum(train_label_freq, 1.0),
        "label_weight": rare_label_weight,
    }).sort_values("original_train_freq")
    sampler_report.to_csv(ARTIFACT_DIR / "weighted_sampler_report.csv", index=False)
    print("Weighted sampler enabled.")
    print("Sample weight min/mean/max:", float(sample_weight.min()), float(sample_weight.mean()), float(sample_weight.max()))
    print("Saved weighted sampler report to:", ARTIFACT_DIR / "weighted_sampler_report.csv")
else:
    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())


## 4. Metrics va threshold helpers

In [ ]:
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def predict_with_threshold_fallback(y_score, threshold=0.5, min_k=1, max_k=3):
    y_pred = np.zeros_like(y_score, dtype=int)
    for i, score_row in enumerate(y_score):
        selected = np.where(score_row >= threshold)[0]
        if len(selected) < min_k:
            selected = np.argsort(score_row)[-min_k:]
        if len(selected) > max_k:
            selected = selected[np.argsort(score_row[selected])[-max_k:]]
        y_pred[i, selected] = 1
    return y_pred

def top_k_predictions(y_score, k=3):
    y_pred = np.zeros_like(y_score, dtype=int)
    top_idx = np.argsort(y_score, axis=1)[:, -k:]
    y_pred[np.arange(len(y_score))[:, None], top_idx] = 1
    return y_pred

def ranking_precision_recall_at_k(y_true, y_score, k=3):
    y_topk = top_k_predictions(y_score, k=k)
    hits = (y_topk * y_true).sum(axis=1)
    return float((hits / k).mean()), float((hits / np.maximum(y_true.sum(axis=1), 1)).mean())

def evaluate_multilabel(y_true, y_pred, y_score=None, model_name="model"):
    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    out = {
        "model": model_name,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_p,
        "macro_precision": macro_p,
        "weighted_precision": weighted_p,
        "micro_recall": micro_r,
        "macro_recall": macro_r,
        "weighted_recall": weighted_r,
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": float((y_true == y_pred).all(axis=1).mean()),
        "avg_true_labels": float(y_true.sum(axis=1).mean()),
        "avg_pred_labels": float(y_pred.sum(axis=1).mean()),
    }
    if y_score is not None:
        p3, r3 = ranking_precision_recall_at_k(y_true, y_score, k=3)
        out["precision_at_3"] = p3
        out["recall_at_3"] = r3
    return out

def tune_global_threshold(y_true, y_score, model_prefix):
    rows = []
    for threshold in THRESHOLD_GRID:
        pred = predict_with_threshold_fallback(y_score, threshold=float(threshold), min_k=MIN_K, max_k=MAX_K)
        row = evaluate_multilabel(y_true.astype(int), pred, y_score, model_name=f"{model_prefix} threshold={threshold:.2f}")
        row["threshold"] = float(threshold)
        rows.append(row)
    result = pd.DataFrame(rows).sort_values("micro_f1", ascending=False).reset_index(drop=True)
    return result, float(result.iloc[0]["threshold"])


## 5. BERT-TextCNN model

In [ ]:
class BertTextCNNForMultilabel(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels,
        filter_sizes=(2, 3, 4, 5),
        num_filters=128,
        dropout=0.35,
        classifier_hidden_dim=512,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if added_tokens > 0:
            self.encoder.resize_token_embeddings(len(tokenizer))
        hidden_size = self.encoder.config.hidden_size
        self.convs = nn.ModuleList([
            nn.Conv1d(hidden_size, num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        cnn_dim = num_filters * len(filter_sizes)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_dim + hidden_size, classifier_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, num_labels),
        )

    def set_encoder_trainable(self, trainable=True):
        for p in self.encoder.parameters():
            p.requires_grad = trainable

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(hidden)
        masked_hidden = hidden * mask
        mean_pooled = masked_hidden.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        conv_input = hidden.transpose(1, 2)
        cnn_features = []
        for conv in self.convs:
            feature = torch.relu(conv(conv_input))
            pooled = torch.max(feature, dim=2).values
            cnn_features.append(pooled)
        cnn_pooled = torch.cat(cnn_features, dim=1)
        features = torch.cat([cnn_pooled, mean_pooled], dim=1)
        return self.classifier(features)

model = BertTextCNNForMultilabel(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    filter_sizes=CNN_FILTER_SIZES,
    num_filters=CNN_NUM_FILTERS,
    dropout=CNN_DROPOUT,
    classifier_hidden_dim=CLASSIFIER_HIDDEN_DIM,
).to(DEVICE)

print(model.__class__.__name__)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Training loop with Kaggle step logs

In [ ]:
train_pos = Y_train.sum(axis=0).astype(np.float32)
train_neg = len(Y_train) - train_pos
pos_weight = (train_neg / np.maximum(train_pos, 1.0)) ** POS_WEIGHT_POWER
pos_weight = np.clip(pos_weight, 1.0, POS_WEIGHT_CAP).astype(np.float32)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

encoder_params = list(model.encoder.parameters())
head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": LEARNING_RATE_ENCODER},
        {"params": head_params, "lr": LEARNING_RATE_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

total_update_steps = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_update_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def predict_scores(data_loader):
    model.eval()
    logits_all = []
    labels_all = []
    for batch in data_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()
        if USE_AMP:
            with torch.cuda.amp.autocast():
                logits = model(input_ids=input_ids, attention_mask=attention_mask)
        else:
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
        logits_all.append(logits.detach().cpu().float().numpy())
        labels_all.append(labels)
    logits_all = np.vstack(logits_all)
    labels_all = np.vstack(labels_all)
    return sigmoid_np(logits_all).astype(np.float32), labels_all.astype(np.float32)

best_val_micro_f1 = -1.0
best_state = None
best_epoch = 0
bad_epochs = 0
global_update_step = 0
train_start = time.time()

print("=" * 100, flush=True)
print("TRAIN BERT-TEXTCNN", flush=True)
print("Model:", MODEL_NAME, flush=True)
print("Train/Val/Test:", len(train_df), len(val_df), len(test_df), flush=True)
print("Max length:", MAX_LENGTH, "Batch:", TRAIN_BATCH_SIZE, "Grad accum:", GRADIENT_ACCUMULATION_STEPS, flush=True)
print("Total update steps:", total_update_steps, "Warmup steps:", warmup_steps, flush=True)
print("=" * 100, flush=True)

for epoch in range(1, NUM_EPOCHS + 1):
    if FREEZE_ENCODER_EPOCHS > 0:
        model.set_encoder_trainable(epoch > FREEZE_ENCODER_EPOCHS)
        print(f"[freeze] epoch={epoch} encoder_trainable={epoch > FREEZE_ENCODER_EPOCHS}", flush=True)

    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    epoch_start = time.time()

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels) / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        running_loss += float(loss.detach().cpu()) * GRADIENT_ACCUMULATION_STEPS

        should_update = (step % GRADIENT_ACCUMULATION_STEPS == 0) or (step == len(train_loader))
        if should_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_update_step += 1

        if step == 1 or step % LOG_EVERY_STEPS == 0 or step == len(train_loader):
            pct = 100.0 * step / len(train_loader)
            avg_loss = running_loss / step
            lr_encoder = optimizer.param_groups[0]["lr"]
            lr_head = optimizer.param_groups[1]["lr"]
            print(
                f"[train-step] epoch={epoch}/{NUM_EPOCHS} batch={step}/{len(train_loader)} ({pct:.1f}%) "
                f"update_step={global_update_step}/{total_update_steps} loss={avg_loss:.6f} "
                f"lr_encoder={lr_encoder:.3g} lr_head={lr_head:.3g} elapsed={(time.time() - train_start) / 60:.1f}m",
                flush=True,
            )

    val_scores, _ = predict_scores(val_loader)
    val_pred_050 = predict_with_threshold_fallback(val_scores, threshold=0.5, min_k=MIN_K, max_k=MAX_K)
    val_metrics_050 = evaluate_multilabel(Y_val.astype(int), val_pred_050, val_scores, model_name="bert_textcnn_val_0.50")
    print(
        f"[eval] epoch={epoch} micro_f1@0.50={val_metrics_050['micro_f1']:.6f} "
        f"macro_f1={val_metrics_050['macro_f1']:.6f} weighted_f1={val_metrics_050['weighted_f1']:.6f} "
        f"p@3={val_metrics_050['precision_at_3']:.6f} r@3={val_metrics_050['recall_at_3']:.6f} "
        f"epoch_time={(time.time() - epoch_start) / 60:.1f}m",
        flush=True,
    )

    if val_metrics_050["micro_f1"] > best_val_micro_f1:
        best_val_micro_f1 = val_metrics_050["micro_f1"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
        torch.save(best_state, ARTIFACT_DIR / "bert_textcnn_best_state.pt")
        print(f"[best] epoch={epoch} best_micro_f1@0.50={best_val_micro_f1:.6f}", flush=True)
    else:
        bad_epochs += 1
        print(f"[early] no_improve_epochs={bad_epochs}/{EARLY_STOPPING_PATIENCE}", flush=True)
        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.", flush=True)
            break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print("Best epoch:", best_epoch, "best validation micro_f1@0.50:", best_val_micro_f1)

tokenizer.save_pretrained(ARTIFACT_DIR / "tokenizer")
torch.save(model.state_dict(), ARTIFACT_DIR / "bert_textcnn_final_state.pt")


## 7. Tune threshold tren validation

In [ ]:
val_scores, _ = predict_scores(val_loader)
threshold_results_val, best_threshold = tune_global_threshold(Y_val, val_scores, "BERT-TextCNN")
metric_columns = [
    "micro_f1",
    "macro_f1",
    "micro_precision",
    "macro_precision",
    "micro_recall",
    "macro_recall",
    "precision_at_3",
    "recall_at_3",
]
display_cols = ["threshold"] + metric_columns
display(threshold_results_val[display_cols].round(3))
print("Best global threshold by validation micro F1:", best_threshold)
threshold_results_val.to_csv(ARTIFACT_DIR / "threshold_results_validation.csv", index=False)
np.save(ARTIFACT_DIR / "val_scores.npy", val_scores)


## 8. Test evaluation

In [ ]:
test_scores, _ = predict_scores(test_loader)

test_best_pred = predict_with_threshold_fallback(test_scores, threshold=best_threshold, min_k=MIN_K, max_k=MAX_K)
test_soc_pred = predict_with_threshold_fallback(test_scores, threshold=SOC_THRESHOLD, min_k=MIN_K, max_k=MAX_K)
test_metrics = pd.DataFrame([
    evaluate_multilabel(Y_test.astype(int), test_best_pred, test_scores, model_name=f"BERT-TextCNN best_threshold={best_threshold:.2f}"),
    evaluate_multilabel(Y_test.astype(int), test_soc_pred, test_scores, model_name=f"BERT-TextCNN SOC threshold={SOC_THRESHOLD:.2f}"),
])
metric_columns = [
    "micro_f1",
    "macro_f1",
    "micro_precision",
    "macro_precision",
    "micro_recall",
    "macro_recall",
    "precision_at_3",
    "recall_at_3",
]
test_metrics = test_metrics[["model"] + metric_columns]
test_metrics[metric_columns] = test_metrics[metric_columns].round(3)
display(test_metrics.set_index("model"))
test_metrics.to_csv(ARTIFACT_DIR / "test_metrics.csv", index=False)
np.save(ARTIFACT_DIR / "test_scores.npy", test_scores)
np.save(ARTIFACT_DIR / "Y_test.npy", Y_test)

per_label_p, per_label_r, per_label_f1, per_label_support = precision_recall_fscore_support(
    Y_test.astype(int), test_best_pred, average=None, zero_division=0
)
per_label_report = pd.DataFrame({
    "label": label_classes,
    "full_support": Y_all.sum(axis=0).astype(int),
    "train_support": Y_train.sum(axis=0).astype(int),
    "val_support": Y_val.sum(axis=0).astype(int),
    "support": per_label_support.astype(int),
    "precision": per_label_p,
    "recall": per_label_r,
    "f1": per_label_f1,
}).sort_values(["f1", "support"])
per_label_report.to_csv(ARTIFACT_DIR / "per_label_report_test_best_threshold.csv", index=False)
print("Saved per-label report to:", ARTIFACT_DIR / "per_label_report_test_best_threshold.csv")

low_f1_labels = per_label_report[per_label_report["f1"] < 0.50].copy()
low_f1_labels = low_f1_labels.rename(columns={"support": "test_support"})
low_f1_display_cols = [
    "label",
    "full_support",
    "train_support",
    "val_support",
    "test_support",
    "precision",
    "recall",
    "f1",
]
low_f1_labels[["precision", "recall", "f1"]] = low_f1_labels[["precision", "recall", "f1"]].round(3)
print("Labels with F1 < 50%:", len(low_f1_labels))
display(low_f1_labels[low_f1_display_cols])
low_f1_labels.to_csv(ARTIFACT_DIR / "labels_f1_below_50.csv", index=False)

run_summary = {
    "model_name": MODEL_NAME,
    "architecture": "BERT-TextCNN",
    "text_col": TEXT_COL,
    "num_labels": int(num_labels),
    "train_samples": int(len(train_df)),
    "validation_samples": int(len(val_df)),
    "test_samples": int(len(test_df)),
    "max_length": int(MAX_LENGTH),
    "cnn_filter_sizes": CNN_FILTER_SIZES,
    "cnn_num_filters": CNN_NUM_FILTERS,
    "best_epoch": int(best_epoch),
    "best_validation_micro_f1_at_0_50": float(best_val_micro_f1),
    "best_threshold": float(best_threshold),
    "use_weighted_sampler": bool(USE_WEIGHTED_SAMPLER),
    "rare_weight_power": float(RARE_WEIGHT_POWER),
    "rare_weight_cap": float(RARE_WEIGHT_CAP),
    "sample_weight_cap": float(SAMPLE_WEIGHT_CAP),
    "pos_weight_power": float(POS_WEIGHT_POWER),
    "pos_weight_cap": float(POS_WEIGHT_CAP),
    "labels_f1_below_50_count": int(len(low_f1_labels)),
    "artifact_dir": str(ARTIFACT_DIR),
}
with open(ARTIFACT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)
print(json.dumps(run_summary, indent=2, ensure_ascii=False))


## 9. Inference helper

In [ ]:
def predict_texts(texts, threshold=None, max_k=3):
    threshold = best_threshold if threshold is None else threshold
    temp_df = pd.DataFrame({TEXT_COL: [str(t) for t in texts]})
    dummy_y = np.zeros((len(temp_df), num_labels), dtype=np.float32)
    temp_dataset = CTIMultilabelDataset(temp_df[TEXT_COL].values, dummy_y, tokenizer, MAX_LENGTH)
    temp_loader = DataLoader(temp_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
    scores, _ = predict_scores(temp_loader)
    y_pred = predict_with_threshold_fallback(scores, threshold=threshold, min_k=1, max_k=max_k)
    rows = []
    for i, score_row in enumerate(scores):
        selected = np.where(y_pred[i] == 1)[0]
        selected = selected[np.argsort(score_row[selected])[::-1]]
        rows.append({
            "text": texts[i],
            "predicted_labels": [label_classes[idx] for idx in selected],
            "scores": [float(score_row[idx]) for idx in selected],
        })
    return pd.DataFrame(rows)

# Example:
# predict_texts(["The adversary used PowerShell to download and execute a payload from a C2 server."])
